In [3]:
# Import knižníc
import pandas as pd
from transformers import pipeline
import re

c:\Users\marti\Diplomovka\SlovakBERT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Mapovanie UPOS na Xpos (prvá pozícia)
upos_to_xpos = {
    "NOUN":  "S",
    "PROPN": "S",
    "PUNCT": "Z",
    "VERB":  "V",
    "ADJ":   "A",
    "ADV":   "D",
    "ADP":   "E",
    "PRON":  "P",
    "DET":   "P",
    "AUX":   "V",
    "CCONJ": "O",
    "SCONJ": "O",
    "PART":  "T",
    "NUM":   "N",
    "INTJ":  "J",
    "SYM":   "X",
    "X":     "X"
}

In [5]:
pipe = pipeline(
    "token-classification",
    model="kinit/slovakbert-pos",
    aggregation_strategy="simple"
)


Device set to use cpu


In [6]:
def read_text_file(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def merge_pipeline_tokens(results):
    # Ak používame aggregation_strategy="simple", výsledky by mali byť už zlúčené.
    return results

def process_token(token, computed_xpos):
    """
    Spracuje token a prípadne ho rozdelí podľa medzier alebo interpunkcie.
    - Ak token obsahuje medzeru (napr. "už skoro"), rozdelí ho na jednotlivé slová a každému priradí computed_xpos.
    - Ak token začína interpunkciou (napr. ",ale"), rozdelí ho na [punctuation, remainder]:
         * Punctuation token dostane UPOS "Z"
         * Remainder token dostane computed_xpos.
    - Ak token končí bodkou a všetky znaky okrem posledného sú číslice (napr. "1."), rozdelí ho na [number, "."]:
         * Number token dostane computed_xpos (napr. podľa mapovania pre čísla)
         * Bodka dostane "Z"
    - Inak vráti token s computed_xpos.
    """
    token = token.strip()
    if " " in token:
        parts = token.split()
        return [(part, part.lower(), computed_xpos) for part in parts]
    if token[0] in {",", ".", ";", ":"} and len(token) > 1:
        punct = token[0]
        remainder = token[1:].strip()
        result = []
        result.append((punct, punct.lower(), "Z"))
        if remainder:
            result.append((remainder, remainder.lower(), computed_xpos))
        return result
    if token.endswith(".") and token[:-1].isdigit():
        number = token[:-1]
        return [(number, number.lower(), computed_xpos), (".", ".", "Z")]
    return [(token, token.lower(), computed_xpos)]

words_data = []
text_file = r"C:\Users\marti\Diplomovka\DiploDiktaty.txt"
lines = read_text_file(text_file)

for line in lines:
    results = pipe(line)
    merged_results = merge_pipeline_tokens(results)
    for res in merged_results:
        token = res["word"].strip()
        if not token:
            continue
        # Získanie UPOS tagu z výsledkov pipeline (skúšame "entity" alebo "entity_group")
        upos_tag = res.get("entity", res.get("entity_group"))
        computed_xpos = upos_to_xpos.get(upos_tag, "X")
        processed_tokens = process_token(token, computed_xpos)
        for tok, lemma, xpos in processed_tokens:
            words_data.append([tok, lemma, xpos])

df = pd.DataFrame(words_data, columns=["Token", "Lemma", "Xpos1"])
df.to_excel("output2.xlsx", index=False, engine="openpyxl")
print(f"Hotovo! Počet tokenov: {len(df)}. Výstup uložený v 'output2.xlsx'.")

Hotovo! Počet tokenov: 1212. Výstup uložený v 'output2.xlsx'.
